<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_06-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 지능시스템 실습 3: LSA 기반 Dense Retrieval

이번 실습에서는 **LSA(Latent Semantic Analysis)** 를 이용하여 문서를 **저차원 실수 벡터**로 바꾸고,
질의와 문서의 **Cosine Similarity**를 계산하여 관련 문서를 검색합니다.

### 학습 목표
- TF-IDF 문서 행렬의 의미를 이해한다.
- SVD를 이용해 고차원 TF-IDF를 저차원 벡터로 줄이는 원리를 이해한다.
- 질의와 문서를 같은 벡터 공간에 표현한다.
- Cosine Similarity로 문서 순위를 계산한다.
- BM25와 Dense Retrieval의 차이를 직관적으로 비교한다.

> 주의: LSA는 현대적인 신경망 기반 임베딩 모델은 아닙니다.  
> 하지만 **문서를 dense vector로 바꾸고 벡터 유사도로 검색한다**는 개념을 이해하기 위한 좋은 교육용 예제입니다.

## 1. 필요한 라이브러리

Colab에는 보통 `scikit-learn`이 설치되어 있으므로 바로 사용할 수 있습니다.

In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import cosine_similarity

## 2. 실습용 문서 만들기

LSA는 여러 문서에서 **함께 등장하는 단어의 패턴**을 이용합니다.

따라서 한두 개의 문서보다, 서로 관련된 주제를 가진 여러 문서를 사용하는 것이 좋습니다.

In [2]:
documents = {
    "doc1": "RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용한다",
    "doc2": "검색 증강 생성은 필요한 자료를 먼저 찾고 그 내용을 바탕으로 답변한다",
    "doc3": "Retrieval 시스템은 질의와 관련된 문서를 찾아 순위를 계산한다",
    "doc4": "BM25는 검색어와 문서에 등장하는 단어의 일치를 이용하는 키워드 검색 방법이다",

    "doc5": "임베딩은 문장이나 문서를 실수 벡터로 표현하는 방법이다",
    "doc6": "Dense Retrieval은 질의와 문서를 벡터로 표현하고 의미적 유사도를 계산한다",
    "doc7": "코사인 유사도는 두 벡터의 방향이 얼마나 비슷한지 측정한다",
    "doc8": "LSA는 행렬 분해를 사용하여 문서를 저차원 의미 공간의 벡터로 표현한다",

    "doc9": "Chunking은 긴 문서를 검색하기 좋은 작은 단위로 나누는 과정이다",
    "doc10": "문서 조각이 너무 크거나 너무 작으면 RAG 검색 품질이 낮아질 수 있다",

    "doc11": "AI Agent는 목표를 달성하기 위해 도구를 선택하고 여러 단계를 수행할 수 있다",
    "doc12": "에이전트는 상황을 판단하고 필요한 도구를 호출하여 작업을 수행한다"
}

for doc_id, text in documents.items():
    print(doc_id, ":", text)

doc1 : RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용한다
doc2 : 검색 증강 생성은 필요한 자료를 먼저 찾고 그 내용을 바탕으로 답변한다
doc3 : Retrieval 시스템은 질의와 관련된 문서를 찾아 순위를 계산한다
doc4 : BM25는 검색어와 문서에 등장하는 단어의 일치를 이용하는 키워드 검색 방법이다
doc5 : 임베딩은 문장이나 문서를 실수 벡터로 표현하는 방법이다
doc6 : Dense Retrieval은 질의와 문서를 벡터로 표현하고 의미적 유사도를 계산한다
doc7 : 코사인 유사도는 두 벡터의 방향이 얼마나 비슷한지 측정한다
doc8 : LSA는 행렬 분해를 사용하여 문서를 저차원 의미 공간의 벡터로 표현한다
doc9 : Chunking은 긴 문서를 검색하기 좋은 작은 단위로 나누는 과정이다
doc10 : 문서 조각이 너무 크거나 너무 작으면 RAG 검색 품질이 낮아질 수 있다
doc11 : AI Agent는 목표를 달성하기 위해 도구를 선택하고 여러 단계를 수행할 수 있다
doc12 : 에이전트는 상황을 판단하고 필요한 도구를 호출하여 작업을 수행한다


## 3. 간단한 한국어 토큰화

이번 실습에서는 원리를 쉽게 보기 위해 **공백 기반 토큰화**를 사용합니다.

실제 한국어 검색 시스템에서는 형태소 분석기나 더 정교한 tokenizer를 사용할 수 있습니다.

In [3]:
def tokenize_korean(text):
    """
    아주 간단한 한국어 tokenizer

    1. 영문은 소문자로 바꿉니다.
    2. 한글, 영문, 숫자만 남깁니다.
    3. 공백을 기준으로 단어를 나눕니다.
    """

    text = text.lower()
    text = re.sub(r"[^0-9a-zA-Z가-힣]+", " ", text)

    return text.split()


sample = "Dense Retrieval은 문서를 벡터로 표현합니다."

print("원문:", sample)
print("토큰:", tokenize_korean(sample))

원문: Dense Retrieval은 문서를 벡터로 표현합니다.
토큰: ['dense', 'retrieval은', '문서를', '벡터로', '표현합니다']


## 4. TF-IDF 행렬 만들기

먼저 각 문서를 **TF-IDF 벡터**로 바꿉니다.

TF-IDF 벡터는 단어 수만큼 차원을 가질 수 있으므로 보통 **고차원이고 sparse(희소)** 합니다.

In [4]:
# 문서 ID와 문장 목록 준비
doc_ids = list(documents.keys())
doc_texts = list(documents.values())

# 사용자 정의 tokenizer를 사용하여 TF-IDF 생성
vectorizer = TfidfVectorizer(
    tokenizer=tokenize_korean,
    token_pattern=None,
    lowercase=False
)

# 문서 -> TF-IDF 행렬
X_tfidf = vectorizer.fit_transform(doc_texts)

print("문서 수:", X_tfidf.shape[0])
print("TF-IDF 차원 수:", X_tfidf.shape[1])
print("TF-IDF 행렬 크기:", X_tfidf.shape)

문서 수: 12
TF-IDF 차원 수: 95
TF-IDF 행렬 크기: (12, 95)


### TF-IDF는 왜 sparse인가?

한 문서에는 전체 단어 중 일부만 등장합니다.

따라서 대부분의 값이 0이 되고, TF-IDF 행렬은 **희소 행렬(sparse matrix)** 이 됩니다.

In [5]:
# 첫 번째 문서의 TF-IDF 벡터를 배열로 확인
first_vector = X_tfidf[0].toarray()[0]

print("전체 차원 수:", len(first_vector))
print("0이 아닌 값의 개수:", np.count_nonzero(first_vector))

전체 차원 수: 95
0이 아닌 값의 개수: 8


## 5. SVD로 차원 줄이기

LSA에서는 SVD 계열의 행렬 분해를 이용해 TF-IDF 공간을 더 작은 차원으로 줄입니다.

예를 들어:

- TF-IDF: 수십~수천 차원
- LSA: 몇 개~수백 개 차원

이렇게 변환하면 각 문서는 **작은 실수 벡터(dense vector)** 로 표현됩니다.

이번 작은 예제에서는 이해를 쉽게 하기 위해 **4차원**으로 줄입니다.

In [6]:
# 사용할 LSA 차원
n_components = 4

# Truncated SVD 생성
svd = TruncatedSVD(
    n_components=n_components,
    random_state=42
)

# TF-IDF -> 저차원 dense vector
X_lsa = svd.fit_transform(X_tfidf)

print("변환 전 TF-IDF 크기:", X_tfidf.shape)
print("변환 후 LSA 크기:", X_lsa.shape)

print()
print("doc1의 LSA 벡터:")
print(np.round(X_lsa[0], 4))

변환 전 TF-IDF 크기: (12, 95)
변환 후 LSA 크기: (12, 4)

doc1의 LSA 벡터:
[ 0.2602 -0.0233  0.011  -0.0668]


## 6. 벡터 정규화

Cosine Similarity를 쉽게 계산하기 위해 각 벡터의 길이를 1로 맞춥니다.

In [7]:
normalizer = Normalizer()

# 각 문서 벡터의 길이를 1로 정규화
X_lsa_norm = normalizer.fit_transform(X_lsa)

print("doc1 정규화 벡터:")
print(np.round(X_lsa_norm[0], 4))

print()
print("doc1 벡터 길이:")
print(np.linalg.norm(X_lsa_norm[0]))

doc1 정규화 벡터:
[ 0.9641 -0.0864  0.0408 -0.2476]

doc1 벡터 길이:
1.0


## 7. 질의를 같은 벡터 공간으로 변환하기

중요한 점은 **문서와 질의가 같은 변환 과정을 사용해야 한다**는 것입니다.

질의도 다음 순서로 변환합니다.

**Query → TF-IDF → SVD → Normalize**

In [8]:
query = "문서를 벡터로 표현해서 비슷한 자료를 찾는 방법"

# 1. 질의를 기존 TF-IDF 공간으로 변환
q_tfidf = vectorizer.transform([query])

# 2. 같은 SVD 모델로 저차원 벡터로 변환
q_lsa = svd.transform(q_tfidf)

# 3. 벡터 길이를 1로 정규화
q_lsa_norm = normalizer.transform(q_lsa)

print("질의:", query)
print()
print("질의 LSA 벡터:")
print(np.round(q_lsa_norm[0], 4))

질의: 문서를 벡터로 표현해서 비슷한 자료를 찾는 방법

질의 LSA 벡터:
[0.9133 0.2049 0.2475 0.2502]


## 8. Cosine Similarity 계산하기

질의 벡터와 모든 문서 벡터의 유사도를 계산합니다.

Cosine Similarity가 클수록 두 벡터의 방향이 비슷합니다.

In [9]:
# 질의와 모든 문서의 cosine similarity 계산
scores = cosine_similarity(
    q_lsa_norm,
    X_lsa_norm
)[0]

# 각 문서의 점수 확인
for doc_id, score in zip(doc_ids, scores):
    print(
        doc_id,
        f"{score:.4f}",
        documents[doc_id]
    )

doc1 0.8110 RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용한다
doc2 0.3952 검색 증강 생성은 필요한 자료를 먼저 찾고 그 내용을 바탕으로 답변한다
doc3 0.6223 Retrieval 시스템은 질의와 관련된 문서를 찾아 순위를 계산한다
doc4 0.4678 BM25는 검색어와 문서에 등장하는 단어의 일치를 이용하는 키워드 검색 방법이다
doc5 0.8303 임베딩은 문장이나 문서를 실수 벡터로 표현하는 방법이다
doc6 0.7789 Dense Retrieval은 질의와 문서를 벡터로 표현하고 의미적 유사도를 계산한다
doc7 -0.2698 코사인 유사도는 두 벡터의 방향이 얼마나 비슷한지 측정한다
doc8 0.8571 LSA는 행렬 분해를 사용하여 문서를 저차원 의미 공간의 벡터로 표현한다
doc9 0.8045 Chunking은 긴 문서를 검색하기 좋은 작은 단위로 나누는 과정이다
doc10 -0.0030 문서 조각이 너무 크거나 너무 작으면 RAG 검색 품질이 낮아질 수 있다
doc11 -0.0874 AI Agent는 목표를 달성하기 위해 도구를 선택하고 여러 단계를 수행할 수 있다
doc12 0.2336 에이전트는 상황을 판단하고 필요한 도구를 호출하여 작업을 수행한다


## 9. 점수가 높은 문서부터 정렬하기

실제 검색 시스템처럼 유사도가 높은 문서를 먼저 보여줍니다.

In [10]:
# 높은 점수 순서의 문서 인덱스
ranking = np.argsort(-scores)

print("질의:", query)
print()

for rank, idx in enumerate(ranking[:5], 1):
    print(
        f"{rank}위",
        doc_ids[idx],
        f"score={scores[idx]:.4f}"
    )
    print(" ", documents[doc_ids[idx]])
    print()

질의: 문서를 벡터로 표현해서 비슷한 자료를 찾는 방법

1위 doc8 score=0.8571
  LSA는 행렬 분해를 사용하여 문서를 저차원 의미 공간의 벡터로 표현한다

2위 doc5 score=0.8303
  임베딩은 문장이나 문서를 실수 벡터로 표현하는 방법이다

3위 doc1 score=0.8110
  RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용한다

4위 doc9 score=0.8045
  Chunking은 긴 문서를 검색하기 좋은 작은 단위로 나누는 과정이다

5위 doc6 score=0.7789
  Dense Retrieval은 질의와 문서를 벡터로 표현하고 의미적 유사도를 계산한다



## 10. 검색 함수 만들기

이제 질의를 넣으면 Top-k 문서를 반환하는 함수로 정리합니다.

In [11]:
def lsa_search(query, top_k=5):
    """
    LSA 기반 문서 검색 함수

    query : 검색 질의
    top_k : 상위 몇 개 문서를 보여줄지
    """

    # 1. 질의를 TF-IDF 벡터로 변환
    q_tfidf = vectorizer.transform([query])

    # 2. 같은 SVD 공간으로 변환
    q_lsa = svd.transform(q_tfidf)

    # 3. 정규화
    q_lsa_norm = normalizer.transform(q_lsa)

    # 4. 문서들과 cosine similarity 계산
    scores = cosine_similarity(
        q_lsa_norm,
        X_lsa_norm
    )[0]

    # 5. 높은 점수 순으로 정렬
    ranking = np.argsort(-scores)

    results = []

    for idx in ranking[:top_k]:
        results.append({
            "문서": doc_ids[idx],
            "유사도": scores[idx],
            "내용": documents[doc_ids[idx]]
        })

    return results

In [12]:
# 검색 함수 실행
query = "질의와 문장을 벡터로 바꾸어 유사도를 계산하는 검색"

results = lsa_search(query, top_k=5)

df = pd.DataFrame(results)

# 보기 좋게 반올림
df["유사도"] = df["유사도"].round(4)

print("질의:", query)
df

질의: 질의와 문장을 벡터로 바꾸어 유사도를 계산하는 검색


,문서,유사도,내용
0,doc6,0.9185,Dense Retrieval은 질의와 문서를 벡터로 표현하고 의미적 유사도를 계산한다
1,doc1,0.8885,RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용한다
2,doc9,0.8832,Chunking은 긴 문서를 검색하기 좋은 작은 단위로 나누는 과정이다
3,doc8,0.8310,LSA는 행렬 분해를 사용하여 문서를 저차원 의미 공간의 벡터로 표현한다
4,doc3,0.8108,Retrieval 시스템은 질의와 관련된 문서를 찾아 순위를 계산한다


## 11. 여러 질의로 실험하기

질의를 바꾸면서 어떤 문서가 높은 순위에 오는지 확인합니다.

In [13]:
queries = [
    "외부 자료를 찾아서 LLM 답변에 사용하는 방법",
    "문장을 벡터로 바꾸어 비슷한 문서를 찾는 방법",
    "긴 문서를 작은 조각으로 나누는 방법",
    "도구를 사용해서 여러 단계의 작업을 수행하는 AI"
]

for query in queries:
    print("=" * 70)
    print("질의:", query)

    results = lsa_search(query, top_k=3)

    for rank, result in enumerate(results, 1):
        print(
            rank,
            result["문서"],
            f"{result['유사도']:.4f}",
            "-",
            result["내용"]
        )

    print()

질의: 외부 자료를 찾아서 LLM 답변에 사용하는 방법
1 doc2 0.9453 - 검색 증강 생성은 필요한 자료를 먼저 찾고 그 내용을 바탕으로 답변한다
2 doc12 0.7808 - 에이전트는 상황을 판단하고 필요한 도구를 호출하여 작업을 수행한다
3 doc4 0.5042 - BM25는 검색어와 문서에 등장하는 단어의 일치를 이용하는 키워드 검색 방법이다

질의: 문장을 벡터로 바꾸어 비슷한 문서를 찾는 방법
1 doc1 0.9756 - RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용한다
2 doc9 0.9733 - Chunking은 긴 문서를 검색하기 좋은 작은 단위로 나누는 과정이다
3 doc8 0.9717 - LSA는 행렬 분해를 사용하여 문서를 저차원 의미 공간의 벡터로 표현한다

질의: 긴 문서를 작은 조각으로 나누는 방법
1 doc1 0.9984 - RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용한다
2 doc9 0.9975 - Chunking은 긴 문서를 검색하기 좋은 작은 단위로 나누는 과정이다
3 doc8 0.9435 - LSA는 행렬 분해를 사용하여 문서를 저차원 의미 공간의 벡터로 표현한다

질의: 도구를 사용해서 여러 단계의 작업을 수행하는 AI
1 doc11 0.8545 - AI Agent는 목표를 달성하기 위해 도구를 선택하고 여러 단계를 수행할 수 있다
2 doc12 0.7155 - 에이전트는 상황을 판단하고 필요한 도구를 호출하여 작업을 수행한다
3 doc10 0.6661 - 문서 조각이 너무 크거나 너무 작으면 RAG 검색 품질이 낮아질 수 있다



## 12. TF-IDF 검색과 LSA 검색 비교하기

이번에는 SVD를 사용하지 않은 **TF-IDF cosine 검색**과  
LSA 기반 검색을 비교합니다.

- TF-IDF: 단어 차원의 sparse vector
- LSA: SVD로 압축한 dense vector

In [14]:
# TF-IDF 벡터도 길이를 정규화
tfidf_normalizer = Normalizer()
X_tfidf_norm = tfidf_normalizer.fit_transform(X_tfidf)


def tfidf_search(query, top_k=5):
    """
    TF-IDF cosine similarity 검색
    """

    q = vectorizer.transform([query])
    q = tfidf_normalizer.transform(q)

    scores = cosine_similarity(
        q,
        X_tfidf_norm
    )[0]

    ranking = np.argsort(-scores)

    results = []

    for idx in ranking[:top_k]:
        results.append({
            "문서": doc_ids[idx],
            "유사도": scores[idx],
            "내용": documents[doc_ids[idx]]
        })

    return results

In [15]:
query = "문장을 숫자 벡터로 표현해서 관련 자료 찾기"

print("질의:", query)

print("\n=== TF-IDF 검색 ===")
for rank, result in enumerate(tfidf_search(query, 3), 1):
    print(
        rank,
        result["문서"],
        f"{result['유사도']:.4f}",
        result["내용"]
    )

print("\n=== LSA 검색 ===")
for rank, result in enumerate(lsa_search(query, 3), 1):
    print(
        rank,
        result["문서"],
        f"{result['유사도']:.4f}",
        result["내용"]
    )

질의: 문장을 숫자 벡터로 표현해서 관련 자료 찾기

=== TF-IDF 검색 ===
1 doc5 0.3197 임베딩은 문장이나 문서를 실수 벡터로 표현하는 방법이다
2 doc6 0.2795 Dense Retrieval은 질의와 문서를 벡터로 표현하고 의미적 유사도를 계산한다
3 doc8 0.2544 LSA는 행렬 분해를 사용하여 문서를 저차원 의미 공간의 벡터로 표현한다

=== LSA 검색 ===
1 doc8 0.9909 LSA는 행렬 분해를 사용하여 문서를 저차원 의미 공간의 벡터로 표현한다
2 doc1 0.9493 RAG는 외부 문서를 검색하여 LLM의 답변 생성에 활용한다
3 doc9 0.9476 Chunking은 긴 문서를 검색하기 좋은 작은 단위로 나누는 과정이다
